# 06 · 量子启发式算法：模拟分叉求解大规模 Max-Cut

对应讲义实验 E · 约 20 分钟 · 前置：04 QAOA 与 VQE 的 Max-Cut 部分（可选）

模拟分叉（Simulated Bifurcation，SB）受量子绝热演化启发，但本身是经典算法：每个变量是一个振子，随控制参数增大分叉到 ±1，符号就是分组。它可以在 CPU、GPU、昇腾 NPU 上大规模并行，今天就能求解实际规模的问题。

**学完本节，你将能够**
- 把 Max-Cut 写成 Ising 模型，调用 `mindquantum.algorithm.qaia` 求解；
- 用随机分组与贪心搜索作为经典基线，客观评价结果。

In [1]:
import sys; sys.path.insert(0, "..")  # 仅参考答案需要：mqlab 位于上一级目录
import time
import numpy as np
from scipy.sparse import coo_matrix
from mindquantum.algorithm.qaia import BSB, DSB
from mqlab import check, data_path

## 1. 数据：Gset 基准图 G22

G22 有 2,000 个顶点、19,990 条边，已知最优割为 13,359。文件首行是“顶点数 边数”，之后每行是 `i j w`（顶点编号从 1 开始）。

In [2]:
i, j, w = np.loadtxt(data_path('G22.txt'), skiprows=1, unpack=True)
i, j = i.astype(int) - 1, j.astype(int) - 1   # 编号从 0 开始
A = coo_matrix((w, (i, j)), shape=(2000, 2000))
A = (A + A.T).tocsr()                          # 无向图：对称化
print(A.shape, A.nnz // 2, '条边')

(2000, 2000) 19990 条边


## 2. 练习 1 · 目标函数

给每个顶点一个自旋 $x_i = \pm 1$ 表示分组。补全 `cut_value(A, x)`，返回割边数，即两端自旋不同的边数：

$$\text{cut}(x) = \sum_{(i,j)\in E} \frac{1 - x_i x_j}{2}.$$

提示：对称矩阵 A 中每条边出现两次；`x @ (A @ x)` 等于 $\sum_{i,j} A_{ij}x_ix_j$。

In [3]:
def cut_value(A, x):
    # 练习 1
    ### BEGIN SOLUTION
    return float((A.sum() - x @ (A @ x)) / 4)
    ### STUDENT: return ...
    ### END SOLUTION

check.cut_value(cut_value)

✅ 通过：3 个随机图的割边数全部正确


## 3. 用离散模拟分叉（DSB）求解

- 把 Max-Cut 写成 Ising 模型：耦合矩阵 $J = -A$，能量最低的自旋构型就是最大割；
- `batch_size=100`：100 条轨迹同时演化，取最好的一条；
- `n_iter=1000`：演化步数；`np.random.seed(0)` 固定随机初值，结果可复现。

In [4]:
np.random.seed(0)                              # 固定随机初值
t0 = time.perf_counter()
solver = DSB(-A, batch_size=100, n_iter=1000)  # J = −A
solver.update()                                # 100 条轨迹并行演化
cuts = solver.calc_cut()
seconds = time.perf_counter() - t0
print(f'best cut = {cuts.max():.0f}，用时 {seconds:.1f} s')

best cut = 13356，用时 4.4 s


### 现场改一改：先预测，再运行

每一处先写下预测：

1. 随机给每个顶点分组（`np.random.choice([-1, 1], size=2000)`）：割边数大约是？　A 约 5,000　B 约 10,000　C 约 13,000（第 4 节会算）
2. `n_iter` 从 100 增加到 3,000：最好的割边数会怎样？　A 基本不变　B 稳步提高，越来越慢　C 先升后降
3. 换成弹道模拟分叉 `BSB`（`n_iter=1000`）：与 DSB 的 13,356 相比？　A 更好　B 更快，但略差　C 无法收敛

第 2、3 处就是练习 2，可以在那里动手。

<details><summary>答案</summary>

1. B：约 10,000。G22 有 19,990 条边，随机分组时每条边被割断的概率是 1/2。
2. B：13,293 → 13,343 → 13,356 → 13,357，越来越接近已知最优 13,359，收益递减。
3. B：BSB 1,000 步得到 13,333，比 DSB 快，但略差；300 步左右就已饱和。
</details>

## 4. 与经典基线比较

`solver.calc_cut(x)` 可以计算任意分组的割边数（x 的形状为 顶点数 × 分组数）。

In [5]:
n = A.shape[0]
rng = np.random.default_rng(0)

def cut_of(x):
    return float(solver.calc_cut(np.asarray(x, dtype=float).reshape(n, 1))[0])

rand = [cut_of(rng.choice([-1, 1], n)) for _ in range(100)]
print(f'随机分组（100 次）：平均 {np.mean(rand):.0f}')

best_ls = 0
for start in range(10):                        # 贪心局部搜索：反复翻转能增加割边数的顶点
    x = rng.choice([-1, 1], n).astype(float)
    while True:
        gain = x * (A @ x)                     # 翻转顶点 i 带来的割边增量
        k = int(np.argmax(gain))
        if gain[k] <= 0:
            break
        x[k] = -x[k]
    best_ls = max(best_ls, cut_of(x))
print(f'贪心局部搜索（10 次取最好）：{best_ls:.0f}')
print(f'模拟分叉 DSB：{cuts.max():.0f}；已知最优：13359')

随机分组（100 次）：平均 9999


贪心局部搜索（10 次取最好）：12858
模拟分叉 DSB：13356；已知最优：13359


模拟分叉达到已知最优值的 99.98%，明显好于随机分组和贪心搜索。它是今天就能落地的“量子思想”；也提醒我们：量子算法要证明价值，必须先赢过这样强的经典对手。

## 5. 练习 2 · 不同算法与步数（开放练习）

把 `DSB` 换成 `BSB`（弹道模拟分叉），并取 `n_iter` = 100、300、1000、3000，比较割边数与耗时。如果有 GPU 或昇腾 NPU，可以设置 `backend='gpu-float32'` 或 `backend='npu-float32'`（分别需要 PyTorch 或 torch_npu），比较速度。

In [6]:
# 练习 2
### BEGIN SOLUTION
for name, cls in [('BSB', BSB), ('DSB', DSB)]:
    for n_iter in [100, 300, 1000, 3000]:
        np.random.seed(0)
        t0 = time.perf_counter()
        s = cls(-A, batch_size=100, n_iter=n_iter)
        s.update()
        print(f'{name}  n_iter = {n_iter:>4}   best cut = {s.calc_cut().max():.0f}   {time.perf_counter() - t0:5.1f} s')
### END SOLUTION

BSB  n_iter =  100   best cut = 13322     0.4 s


BSB  n_iter =  300   best cut = 13332     1.3 s


BSB  n_iter = 1000   best cut = 13333     3.7 s


BSB  n_iter = 3000   best cut = 13330    11.2 s


DSB  n_iter =  100   best cut = 13293     0.5 s


DSB  n_iter =  300   best cut = 13343     1.4 s


DSB  n_iter = 1000   best cut = 13356     4.3 s


DSB  n_iter = 3000   best cut = 13357    13.2 s


## 小结

- 量子启发式算法在经典硬件上运行，不需要量子计算机，今天就能求解大规模组合优化问题。
- 迭代步数越多，结果越好，耗时也线性增长；BSB 更快，DSB 通常更准。
- 同一套接口只需改 `backend` 参数，就能从 CPU 切换到 GPU 或昇腾 NPU。